In [2]:
import sympy as sp

In [3]:
from functools import lru_cache

@lru_cache(maxsize = None)
def get_ckn(k: int, n: int, p):
    if k<0 or k>n:
        return sp.Integer(0)
    if n == 0:
        return sp.Integer(1) if k==0 else sp.Integer(0)
    return get_ckn(k-1, n-1, p)/(2*p) + (k+1)*get_ckn(k+1, n-1, p)


### Implementation of electron-nuclear integrals

In [4]:
x,y,z = sp.symbols('x y z', real = True)
alpha, beta = sp.symbols('alpha beta', real = True, positive = True)
Ax, Ay, Az = sp.symbols('Ax Ay Az', real = True)
Bx, By, Bz = sp.symbols('Bx By Bz', real = True)
Cx, Cy, Cz = sp.symbols('Cx Cy Cz', real = True)

P = alpha + beta
Q = alpha * beta
RAB2 = (Ax-Bx)**2 + (Ay-By)**2 + (Az-Bz)**2
Px = (alpha * Ax + beta * Bx)/(alpha + beta)
Py = (alpha * Ay + beta * By)/(alpha + beta)
Pz = (alpha * Az + beta * Bz)/(alpha + beta)
t_arg = P  * ((Px-Cx)**2 + (Py-Cy)**2 +(Pz-Cz)**2)

In [5]:
class Boys(sp.Function):
    nargs = 2
    def fdiff(self, argindex = 1):
        if argindex == 2:
            return -Boys(self.args[0]+1, self.args[1])
        raise ValueError(argindex)
    
n = sp.symbols('n', integer = True)
F = Boys(n,x)
F.diff(x,1)

-Boys(n + 1, x)

In [6]:
V00 = 2*sp.pi*sp.exp(-Q*RAB2/P)*Boys(0, t_arg)/P
V00

2*pi*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))*Boys(0, (alpha + beta)*((-Cx + (Ax*alpha + Bx*beta)/(alpha + beta))**2 + (-Cy + (Ay*alpha + By*beta)/(alpha + beta))**2 + (-Cz + (Az*alpha + Bz*beta)/(alpha + beta))**2))/(alpha + beta)

In [7]:
def build_derivative_table(base_integral, Lmax, Avars, Bvars):
    Ax, Ay, Az = Avars
    Bx, By, Bz = Bvars
    
    all_idx = [(i, j, L - i - j) for L in range(Lmax+1) 
                               for i in range(L + 1) 
                               for j in range(L+1-i)]


    @lru_cache(maxsize=None)
    def derivative(i,j,k,l,m,n):
        if (i,j,k,l,m,n) == (0,0,0,0,0,0):
            return base_integral
        if i>0:
            return sp.diff(derivative(i-1,j,k,l,m,n),Ax)
        if j>0:
            return sp.diff(derivative(i,j-1,k,l,m,n),Ay)
        if k>0:
            return sp.diff(derivative(i,j,k-1,l,m,n),Az) 
        if l>0:
            return sp.diff(derivative(i,j,k,l-1,m,n),Bx)
        if m>0:
            return sp.diff(derivative(i,j,k,l,m-1,n),By)
        return sp.diff(derivative(i,j,k,l,m,n-1),Bz) 
    
    derivatives_dict = {}
    for (i,j,k) in all_idx:
        for (l,m,n) in all_idx:
            derivatives_dict[(i,j,k,l,m,n)] = derivative(i,j,k,l,m,n)
    return derivatives_dict

In [8]:
Lmax = 1
derivatives_dict = build_derivative_table(V00, Lmax, (Ax,Ay,Az), (Bx,By,Bz))

In [9]:
for key,value in derivatives_dict.items():
    print(key, value)

(0, 0, 0, 0, 0, 0) 2*pi*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))*Boys(0, (alpha + beta)*((-Cx + (Ax*alpha + Bx*beta)/(alpha + beta))**2 + (-Cy + (Ay*alpha + By*beta)/(alpha + beta))**2 + (-Cz + (Az*alpha + Bz*beta)/(alpha + beta))**2))/(alpha + beta)
(0, 0, 0, 0, 0, 1) -2*pi*alpha*beta*(-2*Az + 2*Bz)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))*Boys(0, (alpha + beta)*((-Cx + (Ax*alpha + Bx*beta)/(alpha + beta))**2 + (-Cy + (Ay*alpha + By*beta)/(alpha + beta))**2 + (-Cz + (Az*alpha + Bz*beta)/(alpha + beta))**2))/(alpha + beta)**2 - 4*pi*beta*(-Cz + (Az*alpha + Bz*beta)/(alpha + beta))*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))*Boys(1, (alpha + beta)*((-Cx + (Ax*alpha + Bx*beta)/(alpha + beta))**2 + (-Cy + (Ay*alpha + By*beta)/(alpha + beta))**2 + (-Cz + (Az*alpha + Bz*beta)/(alpha + beta))**2))/(alpha + beta)
(0, 0, 0, 0, 1, 0) -2*pi*alpha*beta*(-2*Ay + 2*By)*exp(-alpha*beta*((Ax - Bx)

In [10]:
def get_integral_expressions(integral_indices, derivatives_dict, alpha, beta):
    integral_expressions = {}
    for (i,j,k,l,m,n) in integral_indices:

        ci = [get_ckn(o, i, alpha) for o in range(i+1)]
        cj = [get_ckn(o, j, alpha) for o in range(j+1)]
        ck = [get_ckn(o, k, alpha) for o in range(k+1)]
        cl = [get_ckn(o, l, beta) for o in range(l+1)]
        cm = [get_ckn(o, m, beta) for o in range(m+1)]
        cn = [get_ckn(o, n, beta) for o in range(n+1)]

        expr = 0
        for o, co in enumerate(ci):
            for p, cp in enumerate(cj):
                for q, cq in enumerate(ck):

                    for r, cr in enumerate(cl):
                        for s, cs in enumerate(cm):
                            for t, ct in enumerate(cn):
                                expr += co*cp*cq*cr*cs*ct * derivatives_dict[(o, p, q, r, s, t)]
        integral_expressions[(i,j,k,l,m,n)] = expr.simplify()
    return integral_expressions



In [11]:
def l_to_ijk(L):
    IJK = []
    for I in range(L, -1, -1):
        for J in range(L - I, -1, -1):
            IJK.append((I, J, L - I - J))
    return sorted(IJK, reverse = True)

integral_indices = []
ijk = [t for L in range(Lmax+1) for t in l_to_ijk(L)]
for i in ijk:
    for j in ijk:
        integral_indices.append(i+j)
integral_indices

[(0, 0, 0, 0, 0, 0),
 (0, 0, 0, 1, 0, 0),
 (0, 0, 0, 0, 1, 0),
 (0, 0, 0, 0, 0, 1),
 (1, 0, 0, 0, 0, 0),
 (1, 0, 0, 1, 0, 0),
 (1, 0, 0, 0, 1, 0),
 (1, 0, 0, 0, 0, 1),
 (0, 1, 0, 0, 0, 0),
 (0, 1, 0, 1, 0, 0),
 (0, 1, 0, 0, 1, 0),
 (0, 1, 0, 0, 0, 1),
 (0, 0, 1, 0, 0, 0),
 (0, 0, 1, 1, 0, 0),
 (0, 0, 1, 0, 1, 0),
 (0, 0, 1, 0, 0, 1)]

In [12]:
integral_expressions = get_integral_expressions(integral_indices, derivatives_dict, alpha, beta)

In [13]:
for key,value in integral_expressions.items():
    print(key, value)

(0, 0, 0, 0, 0, 0) 2*pi*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))*Boys(0, ((-Ax*alpha - Bx*beta + Cx*(alpha + beta))**2 + (-Ay*alpha - By*beta + Cy*(alpha + beta))**2 + (-Az*alpha - Bz*beta + Cz*(alpha + beta))**2)/(alpha + beta))/(alpha + beta)
(0, 0, 0, 1, 0, 0) 2*pi*(alpha*(Ax - Bx)*Boys(0, ((-Ax*alpha - Bx*beta + Cx*(alpha + beta))**2 + (-Ay*alpha - By*beta + Cy*(alpha + beta))**2 + (-Az*alpha - Bz*beta + Cz*(alpha + beta))**2)/(alpha + beta)) + (-Ax*alpha - Bx*beta + Cx*(alpha + beta))*Boys(1, ((-Ax*alpha - Bx*beta + Cx*(alpha + beta))**2 + (-Ay*alpha - By*beta + Cy*(alpha + beta))**2 + (-Az*alpha - Bz*beta + Cz*(alpha + beta))**2)/(alpha + beta)))*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**2
(0, 0, 0, 0, 1, 0) 2*pi*(alpha*(Ay - By)*Boys(0, ((-Ax*alpha - Bx*beta + Cx*(alpha + beta))**2 + (-Ay*alpha - By*beta + Cy*(alpha + beta))**2 + (-Az*alpha - Bz*beta + Cz*(alpha + beta))**2)/(alpha + beta)) + 

In [14]:
(integral_expressions[(0,0,0,0,0,0)])

2*pi*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))*Boys(0, ((-Ax*alpha - Bx*beta + Cx*(alpha + beta))**2 + (-Ay*alpha - By*beta + Cy*(alpha + beta))**2 + (-Az*alpha - Bz*beta + Cz*(alpha + beta))**2)/(alpha + beta))/(alpha + beta)

In [15]:
Dx, Dy, Dz, P, Q, RAB2 = sp.symbols(' Dx Dy Dz P Q RAB2')
PCx, PCy, PCz = sp.symbols('PCx PCy PCz', real = True)
Qx, Qy, Qz = sp.symbols('Qx Qy Qz', real = True)
P, Q = sp.symbols('P Q', real = True)
u = sp.symbols("u", nonnegative = True, real = True)
KAB = sp.symbols ('KAB', positive = True, real = True)
subsdict = {
    alpha + beta: P,
    alpha*beta: Q,
    Ax-Bx:Dx,
    Ay-By:Dy,
    Az-Bz:Dz,
    (Ax-Bx)**2 + (Ay-By)**2 + (Az-Bz)**2:RAB2,
    (alpha * Ax + beta * Bx)/(alpha + beta): Px,
    (alpha * Ay + beta * By)/(alpha + beta): Py,
    (alpha * Az + beta * Bz)/(alpha + beta): Pz, 
    alpha*Ax + beta*Bx - (alpha + beta)*Cx: Qx,
    alpha*Ay + beta*By - (alpha + beta)*Cy: Qy,
    alpha*Az + beta*Bz - (alpha + beta)*Cz: Qz,
    Qx/P : PCx,
    Qy/P : PCy,
    Qz/P : PCz,
    ((-Ax*alpha - Bx*beta + Cx*(alpha + beta))**2 + 
     (-Ay*alpha - By*beta + Cy*(alpha + beta))**2 + 
     (-Az*alpha - Bz*beta + Cz*(alpha + beta))**2)/(alpha + beta): u,
     sp.exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta)): KAB

}

In [16]:
for key, value in integral_expressions.items():
    integral_expressions[key] = sp.simplify(value.subs(subsdict, simultaneous = True))

In [17]:
for key, value in integral_expressions.items():
    print(f"Integral {key}: {value}")

Integral (0, 0, 0, 0, 0, 0): 2*pi*KAB*Boys(0, u)/P
Integral (0, 0, 0, 1, 0, 0): 2*pi*KAB*(Dx*alpha*Boys(0, u) - Qx*Boys(1, u))/P**2
Integral (0, 0, 0, 0, 1, 0): 2*pi*KAB*(Dy*alpha*Boys(0, u) - Qy*Boys(1, u))/P**2
Integral (0, 0, 0, 0, 0, 1): 2*pi*KAB*(Dz*alpha*Boys(0, u) - Qz*Boys(1, u))/P**2
Integral (1, 0, 0, 0, 0, 0): -2*pi*KAB*(Dx*beta*Boys(0, u) + Qx*Boys(1, u))/P**2
Integral (1, 0, 0, 1, 0, 0): pi*KAB*(-2*Dx**2*Q*Boys(0, u) - 2*Dx*Qx*alpha*Boys(1, u) + 2*Dx*Qx*beta*Boys(1, u) + P*(Boys(0, u) - Boys(1, u)) + 2*Qx**2*Boys(2, u))/P**3
Integral (1, 0, 0, 0, 1, 0): 2*pi*KAB*(-Dx*Dy*Q*Boys(0, u) + Dx*Qy*beta*Boys(1, u) - Dy*Qx*alpha*Boys(1, u) + Qx*Qy*Boys(2, u))/P**3
Integral (1, 0, 0, 0, 0, 1): 2*pi*KAB*(-Dx*Dz*Q*Boys(0, u) + Dx*Qz*beta*Boys(1, u) - Dz*Qx*alpha*Boys(1, u) + Qx*Qz*Boys(2, u))/P**3
Integral (0, 1, 0, 0, 0, 0): -2*pi*KAB*(Dy*beta*Boys(0, u) + Qy*Boys(1, u))/P**2
Integral (0, 1, 0, 1, 0, 0): 2*pi*KAB*(-Dx*Dy*Q*Boys(0, u) - Dx*Qy*alpha*Boys(1, u) + Dy*Qx*beta*Boys(1, u) +

In [23]:
args = set()

for expr in integral_expressions.values():
    args.update(expr.free_symbols)

args = sorted(args, key = lambda x: x.name)
args = ["i","j","k","l","m","n"] + list(args)
args = ", ".join(str(arg) for arg in args)
args

'i, j, k, l, m, n, Dx, Dy, Dz, KAB, P, Q, Qx, Qy, Qz, alpha, beta, u'

In [25]:
from sympy.printing.numpy import NumPyPrinter, _known_functions_numpy, _known_constants_numpy

In [26]:
printer = NumPyPrinter()
printer._module = "np"
printer.known_functions = {k: f"np.{v}" for k,v in _known_functions_numpy.items()}
printer.known_functions["Boys"] = "Boys"
printer.known_constants = {k: f"np.{v}" for k,v in _known_constants_numpy.items()}

In [27]:
print(printer.known_functions)

{'acos': 'np.arccos', 'acosh': 'np.arccosh', 'asin': 'np.arcsin', 'asinh': 'np.arcsinh', 'atan': 'np.arctan', 'atan2': 'np.arctan2', 'atanh': 'np.arctanh', 'ceiling': 'np.ceil', 'cos': 'np.cos', 'cosh': 'np.cosh', 'exp': 'np.exp', 'expm1': 'np.expm1', 'floor': 'np.floor', 'hypot': 'np.hypot', 'isinf': 'np.isinf', 'isnan': 'np.isnan', 'log': 'np.log', 'ln': 'np.log', 'log10': 'np.log10', 'log1p': 'np.log1p', 'log2': 'np.log2', 'sin': 'np.sin', 'sinh': 'np.sinh', 'Sqrt': 'np.sqrt', 'tan': 'np.tan', 'tanh': 'np.tanh', 'exp2': 'np.exp2', 'sign': 'np.sign', 'logaddexp': 'np.logaddexp', 'logaddexp2': 'np.logaddexp2', 'Boys': 'Boys'}


In [30]:
def write_oneel_module(path, name = "S", integral_expressions = None, parameter_list = None):
    lines = ["import numpy as np",
              "from .boys import Boys",
             "from numba import njit",
             "@njit(cache = True, fastmath = True)",
             f"def {name}({parameter_list}):"]
    for key, value in integral_expressions.items():
        lines.append(f"    if (i, j, k, l, m, n) == {key}:")
        lines.append(f"        return {printer.doprint(value)}")
    
    with open(path, "w", encoding = "utf-8") as f:
        f.write("\n".join(lines))


In [31]:
from pathlib import Path
path = Path.cwd() / "V.py" 
write_oneel_module(path, name = "V", integral_expressions= integral_expressions, parameter_list = args)

In [32]:
Path.cwd()

PosixPath('/Users/rolandmitric/MASTER_PROGRAMMING_2026/live_notebooks')